# ReviewPulse: Sentiment Analysis & Model Evaluation
This notebook outlines the Exploratory Data Analysis (EDA), model training (fine-tuning DistilBERT on CPU vs. falling back to a pretrained student pipeline due to time limits), and rigorous evaluation for the product reviews sentiment classifier.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plots style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Exploratory Data Analysis (EDA)
First, let's load our sampled dataset (`data/reviews_sample.csv`) which contains 3,000 reviews (stratified with 1,000 reviews for each sentiment category: positive, neutral, negative) mapped from Amazon Fine Food reviews star ratings.

In [ ]:
data_path = "../data/reviews_sample.csv"
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Loaded dataset with {len(df)} rows.")
    print(df.head())
else:
    print("Sample dataset reviews_sample.csv not found! Please run the download script.")

### 1.1 Sentiment Distribution
Let's confirm the balanced dataset split across the mapped sentiments.

In [ ]:
print(df['sentiment'].value_counts())
sns.countplot(data=df, x='sentiment', palette='viridis')
plt.title('Sample Dataset Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

### 1.2 Star Ratings Distribution
Let's see how the raw star ratings are distributed across our samples. 
Recall that:
- Stars 4-5 -> Positive
- Star 3 -> Neutral
- Stars 1-2 -> Negative

In [ ]:
print(df['stars'].value_counts().sort_index())
sns.countplot(data=df, x='stars', palette='magma')
plt.title('Star Ratings Distribution')
plt.xlabel('Stars')
plt.ylabel('Count')
plt.show()

### 1.3 Review Text Length Analysis
Let's check the length distribution (number of words and characters) of the review texts. This informs tokenization decisions (like `max_length`).

In [ ]:
df['char_length'] = df['review_text'].fillna("").apply(len)
df['word_length'] = df['review_text'].fillna("").apply(lambda x: len(x.split()))

print("Review Length Summary Metrics:")
print(df[['char_length', 'word_length']].describe())

# Plot distribution of word lengths
sns.histplot(data=df, x='word_length', bins=50, kde=True, color='purple')
plt.title('Distribution of Review Word Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Density')
plt.xlim(0, 300) # Limit x-axis to zoom in on density
plt.show()

## 2. Model Training & Validation (Option B vs. Fallback Option A)
Fine-tuning a transformer on CPU is a slow process. To balance quality and interview prep constraints, we set up a hard time-boxed Trainer (using `TimeLimitCallback` limited to 10 minutes). If training succeeded within the time-box, it saved the weights. Otherwise, it aborted and activated the fallback pipeline using a pretrained 3-class student model (`lxyuan/distilbert-base-multilingual-cased-sentiments-student`).

In [ ]:
# Let's check which path was chosen
fallback_path = "../models/fallback_active.txt"
if os.path.exists(fallback_path):
    with open(fallback_path, "r") as f:
        model_used = f.read().strip()
    print(f"Fallback Active: Using Pretrained HF pipeline model '{model_used}'")
else:
    print("Fine-tuned model successfully built locally at 'models/sentiment_model'!")

## 3. Quantitative Model Evaluation
We evaluate the final model (fine-tuned or fallback) on the hold-out test set (500 reviews) which was stratified to ensure representative labels.

In [ ]:
# Print classification report generated by python script
report_path = "../evaluation/classification_report.txt"
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        print(f.read())
else:
    print("Classification report not found. Run scripts/train_and_eval.py first!")

### 3.1 Confusion Matrix
Let's load the confusion matrix plot.

In [ ]:
from PIL import Image
cm_path = "../evaluation/confusion_matrix.png"
if os.path.exists(cm_path):
    img = Image.open(cm_path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Confusion matrix image not found.")

## 4. Qualitative Error Analysis
Inspecting raw errors is critical for junior DS interviews as it demonstrates that you understand the data, know model limits, and don't trust metrics blindly. Common error types include sarcasm, mixed sentiment, and star rating vs. actual sentiment discrepancy.

In [ ]:
errors_path = "../evaluation/misclassified_examples.md"
if os.path.exists(errors_path):
    with open(errors_path, "r") as f:
        # Print first 30 lines of errors analysis
        lines = f.readlines()
        print("".join(lines[:45]))
else:
    print("Error analysis file not found.")